# Clase 5 · Reto: las cancelaciones de un hotel

Eres analista de datos de un **hotel**. Cada cancelación de última hora deja una habitación vacía, así que la dirección quiere un modelo que **avise de las reservas con riesgo de cancelarse** (`cancelada` = 1) para poder actuar a tiempo.

Te dan 1 200 reservas. Los datos traen **fechas** (que un modelo no puede usar tal cual), **texto**, **números** y **nulos**, y solo se cancelan unas **17 de cada 100** reservas.

**Reglas del juego**

1. **Escribe tú el código.** Puedes usar la documentación o la IA para entender una función o un error, pero necesitas saber qué hace cada línea.
2. **Cada ejercicio termina con tu interpretación** en una celda de texto: 1 o 2 frases con *tu* conclusión.
3. Todo el preprocesado va **dentro de un pipeline** (nada de calcular medias a mano antes de partir).

**Tiempo orientativo:** el núcleo (niveles 1 a 4) son unos **25 minutos**. Si terminas antes, las ampliaciones te esperan. **No hace falta llevarte nada a casa.**

| Nivel | Contenido | Tiempo |
|---|---|---|
| 1 | Las fechas | 4 min |
| 2 | Crear variables | 8 min |
| 3 | El pipeline y el desbalanceo | 10 min |
| 4 | Informe | 3 min |
| Ampliación | Para quien termine antes | libre |

> Si te falta `scikit-learn`, ejecuta `%pip install scikit-learn`.

## Preparación

In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import recall_score, precision_score, confusion_matrix

df = pd.read_csv("reservas_hotel.csv")
print("Cargado:", df.shape[0], "filas y", df.shape[1], "columnas")
df.head()

Cargado: 1200 filas y 13 columnas


,id_reserva,fecha_reserva,fecha_llegada,fecha_salida,adultos,ninos,tipo_habitacion,regimen,canal,pais,precio_noche,peticiones_especiales,cancelada
0,R-00001,2026-04-26,2026-08-19,2026-08-22,2,0,Estándar,Desayuno,Web,España,116.26,3,0
1,R-00002,2025-11-07,2026-01-03,2026-01-06,3,1,Estándar,Desayuno,Web,Francia,92.67,4,0
2,R-00003,2026-01-04,2026-02-24,2026-02-26,1,0,Estándar,Solo alojamiento,Teléfono,Otros,92.97,2,0
3,R-00004,2026-01-14,2026-05-21,2026-05-23,2,0,Estándar,Desayuno,Empresa,Reino Unido,81.44,4,0
4,R-00005,2026-02-14,2026-04-14,2026-04-19,1,0,Estándar,Media pensión,Web,España,85.99,0,1


## Nivel 1 · Las fechas

### Ejercicio 1 · Primer vistazo

Muestra los tipos de cada columna (`info`), los nulos por columna y el **porcentaje de reservas canceladas**. Calcula también cuánto acertaría un «modelo» que **siempre dice que no se cancela**.

*Interpretación:* ¿por qué las fechas no se pueden usar tal como están? ¿Qué te dice el modelo trivial sobre lo que tiene que superar tu modelo?

In [2]:
df.head()
df.dtypes
nulos = df["fecha_reserva"].isnull().sum()
print(nulos)
todos_nulos = df.isnull().sum()
print(todos_nulos)
porcentaje_canceladas = (df["cancelada"].mean() *100).round(1)
print("El porcentaje de canceladas es: ",porcentaje_canceladas)
acierto_trivial = 100 - porcentaje_canceladas
print("El modelo trivial acierta el", acierto_trivial, "%")


0
id_reserva                0
fecha_reserva             0
fecha_llegada             0
fecha_salida              0
adultos                   0
ninos                     0
tipo_habitacion           0
regimen                   0
canal                     0
pais                     50
precio_noche             70
peticiones_especiales     0
cancelada                 0
dtype: int64
El porcentaje de canceladas es:  17.0
El modelo trivial acierta el 83.0 %


Las columnas fecha_reserva, fecha_llegada y fecha_salida son de tipo str, es decir texto y el modelo no puede hacer cuentas con un texto como "2026-04-26" por eso hay que convertirlas a fecha y sacar información útil.



### Ejercicio 2 · Convertir las fechas

Convierte `fecha_reserva`, `fecha_llegada` y `fecha_salida` a tipo fecha con `pd.to_datetime`. Comprueba los tipos.

In [3]:
fecha_reserva = pd.to_datetime(df["fecha_reserva"])
fecha_llegada = pd.to_datetime(df["fecha_llegada"])
fecha_salida = pd.to_datetime(df["fecha_salida"])

*Mi interpretación:* escribe aquí 1 o 2 frases.

## Nivel 2 · Crear variables

### Ejercicio 3 · Cuatro variables nuevas

Crea estas cuatro columnas:

- `antelacion_dias`: días entre la reserva y la llegada.
- `noches`: días entre la llegada y la salida.
- `huespedes`: adultos más niños.
- `fin_de_semana`: 1 si la llegada es **viernes o sábado**, 0 si no.

*Pistas:* restar dos fechas da un intervalo; `.dt.days` lo convierte en número. `.dt.day_of_week` da el día de la semana (lunes = 0, domingo = 6) y `isin([4, 5])` comprueba si es viernes o sábado.

In [4]:


df["antelacion_dias"] = (pd.to_datetime(df["fecha_llegada"]) - pd.to_datetime(df["fecha_reserva"])).dt.days
df["noches"] = (pd.to_datetime(df["fecha_salida"]) - pd.to_datetime(df["fecha_llegada"])).dt.days
df["huespedes"] = df["adultos"] + df["ninos"]
df["fin_de_semana"] = pd.to_datetime(df["fecha_llegada"]).dt.day_of_week.isin([4, 5]).astype(int)

df[["antelacion_dias", "noches", "huespedes", "fin_de_semana"]].head()

,antelacion_dias,noches,huespedes,fin_de_semana
0,115,3,2,0
1,57,3,4,1
2,51,2,1,0
3,127,2,2,0
4,59,5,1,0


Contra más noches y más huéspedes, parece que hay más posibilidad de cancelación, hay que seguir investigando

### Ejercicio 4 · ¿Aportan algo?

Comprueba si dos de las variables nuevas tienen relación con la cancelación:

1. El porcentaje de cancelación según `fin_de_semana`.
2. El porcentaje de cancelación según la antelación, agrupada en tramos (por ejemplo, 0-30, 31-60, 61-90 y más de 90 días) con `pd.cut`.

*Interpretación:* ¿qué patrón ves? ¿Cuál de las dos variables parece más útil?

In [5]:
fin_de_semana = (df.groupby("fin_de_semana")["cancelada"].mean() * 100).round(1)
print("Porcentaje de cancelacion: ", fin_de_semana)

df["tramo_antelacion"] = pd.cut(df["antelacion_dias"], bins=[-1, 30 , 60, 90, np.inf], labels=["0-30", "31-60", "61-90", ">90"])

pct_antelacion = (df.groupby("tramo_antelacion")["cancelada"].mean() * 100).round(1)
print("Porcentaje de cancelación")
print(pct_antelacion)


Porcentaje de cancelacion:  fin_de_semana
0    13.2
1    26.9
Name: cancelada, dtype: float64
Porcentaje de cancelación
tramo_antelacion
0-30      5.8
31-60     9.2
61-90    21.6
>90      48.5
Name: cancelada, dtype: float64


Cuanta más antelacioón tiene la reserva, más probqabilidades de que se cancela, algo que es lógico

### Ejercicio 5 · Descartar

Descarta las columnas que ya no sirven: `id_reserva` (un identificador) y las tres fechas (ya hemos sacado de ellas lo que importa).

*Interpretación:* ¿por qué descartar `id_reserva` en lugar de dejarla?

In [ ]:
# TU CÓDIGO AQUÍ
df = df.drop(columns=["id_reserva", "fecha_reserva", "fecha_llegada", "fecha_salida", "tramo_antelacion"])
df.head()

,fecha_reserva,fecha_llegada,fecha_salida,adultos,ninos,tipo_habitacion,regimen,canal,pais,precio_noche,peticiones_especiales,cancelada,antelacion_dias,noches,huespedes,fin_de_semana,tramo_antelacion
0,2026-04-26,2026-08-19,2026-08-22,2,0,Estándar,Desayuno,Web,España,116.26,3,0,115,3,2,0,>90
1,2025-11-07,2026-01-03,2026-01-06,3,1,Estándar,Desayuno,Web,Francia,92.67,4,0,57,3,4,1,31-60
2,2026-01-04,2026-02-24,2026-02-26,1,0,Estándar,Solo alojamiento,Teléfono,Otros,92.97,2,0,51,2,1,0,31-60
3,2026-01-14,2026-05-21,2026-05-23,2,0,Estándar,Desayuno,Empresa,Reino Unido,81.44,4,0,127,2,2,0,>90
4,2026-02-14,2026-04-14,2026-04-19,1,0,Estándar,Media pensión,Web,España,85.99,0,1,59,5,1,0,31-60


Id_reserva no le aporta al modelo una buena información, más bien, lo puede perturbar

## Nivel 3 · El pipeline y el desbalanceo

### Ejercicio 6 · Particionar

Separa `y` (`cancelada`) y `X` (el resto). Parte en entrenamiento (75 %) y prueba (25 %) con `stratify=y` y `random_state=42`. Define las listas de columnas numéricas y categóricas.

*Interpretación:* ¿cuántas reservas canceladas hay en la prueba? ¿Es mucho o poco para fiarse de un recall?

In [18]:
# TU CÓDIGO AQUÍ
y = df["cancelada"]
X = df.drop(columns="cancelada")

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, stratify=y, random_state=42)

numericas_cols = ["adultos", "ninos", "precio_noche", "peticiones_especiales",
            "antelacion_dias", "noches", "huespedes", "fin_de_semana"]

categorica_cols = ["tipo_habitacion", "regimen", "canal", "pais"]

print("Filas de prueba: ", len(y_test))
print("Cuantas Canceladas hay en la prueba??", y_test.sum())

Filas de prueba:  300
Cuantas Canceladas hay en la prueba?? 51


Pocas canceladas en la prueba, en el recall, si un día me varía, puede ser por pocas reservas

### Ejercicio 7 · Tu primer pipeline

Construye un pipeline con un `ColumnTransformer`:

- Numéricas: imputar con la mediana y escalar.
- Categóricas: imputar con el valor más frecuente y *one-hot* (`handle_unknown="ignore"`).

Termina con una `LogisticRegression(max_iter=1000)`. Entrénalo (`fit`) y calcula, en la prueba, los **aciertos**, el **recall** (`recall_score`) y la **precisión** (`precision_score`).

*Interpretación:* compara los aciertos con el modelo trivial (83 %). ¿Es bueno tu modelo? ¿Qué te dice el recall?

In [ ]:
# TU CÓDIGO AQU
num = Pipeline([("Imputar", SimpleImputer(strategy="median")),
                ("escalar", StandardScaler())])

cat = Pipeline([("imputar", SimpleImputer(strategy="most_frequent")),
               ("onehot", OneHotEncoder(handle_unknown="ignore"))])

preprocesado = ColumnTransformer([
    ("num", num, numericas_cols),
    ("cat", cat, categorica_cols)
])

modelo = Pipeline([
    ("prep", preprocesado),
    ("clf", LogisticRegression(max_iter=1000))
])

modelo.fit(X_train, y_train)

print("Aciertos: ", round(modelo.score(X_test, y_test) * 100, 1), "%")
print("Recall: ", round(recall_score(y_test, y_pred) * 100, 1), "%"
)

*Mi interpretación:* escribe aquí 1 o 2 frases.

### Ejercicio 8 · Equilibrar

Vuelve a entrenar el mismo pipeline con `class_weight="balanced"` en la regresión logística. Calcula de nuevo aciertos, recall y precisión.

*Interpretación:* ¿qué ha ganado y qué ha perdido? Si fueras el director del hotel, **¿qué versión elegirías y por qué?**

In [9]:
# TU CÓDIGO AQUÍ

*Mi interpretación:* escribe aquí 1 o 2 frases.

## Nivel 4 · Informe

### Ejercicio 9 · Informe

Escribe, en una celda de texto, tu informe para la dirección del hotel (3 o 4 frases):

- Qué **variables nuevas** has creado y cuál crees que ayuda más.
- Qué modelo **recomiendas** (normal o equilibrado) y **por qué**, con sus aciertos y su recall.

*Escribe aquí tu respuesta.*

## Ampliación (para quien termine antes)

No es obligatoria, pero **sube tu nota de participación**. Hazlas en el orden que quieras.

### Ampliación A · ¿Ayudaron de verdad las variables nuevas?

Compara con `cross_val_score(..., cv=5)` el mismo pipeline **solo con las columnas originales** (`adultos`, `ninos`, `precio_noche`, `peticiones_especiales` y las categóricas) y **con las nuevas**.

*Interpretación:* ¿cuánto mejora la nota?

In [10]:
# TU CÓDIGO AQUÍ

*Mi interpretación:* escribe aquí 1 o 2 frases.

### Ampliación B · Un logaritmo dentro del pipeline

El precio por noche es asimétrico. Añade al pipeline numérico un `FunctionTransformer(np.log1p)` para `precio_noche` (como un grupo aparte dentro del `ColumnTransformer`) y compara con validación cruzada.

*Interpretación:* ¿cambia algo?

In [11]:
# TU CÓDIGO AQUÍ

*Mi interpretación:* escribe aquí 1 o 2 frases.

### Ampliación C · Cambiar de modelo en una línea

Sustituye la regresión logística por un `DecisionTreeClassifier(max_depth=4, random_state=42)` y por un `KNeighborsClassifier()` manteniendo el mismo preprocesado. Compara con validación cruzada.

*Interpretación:* ¿qué modelo gana? ¿Qué tiene de bueno un pipeline para hacer esto?

In [12]:
# TU CÓDIGO AQUÍ

*Mi interpretación:* escribe aquí 1 o 2 frases.

### Ampliación D · Guardar y usar el pipeline

Guarda tu mejor pipeline con `joblib.dump` y cárgalo con `joblib.load`. Predice la probabilidad de cancelación de una reserva nueva que inventes (una fila en bruto, con las mismas columnas que `X`).

*Interpretación:* ¿por qué no necesitas repetir el preprocesado?

In [13]:
# TU CÓDIGO AQUÍ

*Mi interpretación:* escribe aquí 1 o 2 frases.

### Ampliación E · Mini-reto

**¿Quién consigue el mejor recall sin bajar del 75 % de aciertos en la prueba?** Puedes combinar lo que quieras: variables nuevas (por ejemplo `mes_llegada` o `tiene_ninos`), modelos distintos, `class_weight`, sobremuestrear **solo el entrenamiento**… Apunta tu mejor resultado.

*Interpretación:* ¿qué te ha funcionado y qué no?

In [14]:
# TU CÓDIGO AQUÍ

*Mi interpretación:* escribe aquí 1 o 2 frases.